In [1]:
! pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
! pip install pandas requests pillow tqdm datasets transformers hf_xet


Tue Sep 30 16:12:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import BlipProcessor, BlipForConditionalGeneration
import math

2025-09-30 15:57:00.926575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759247821.110098      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759247821.163560      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
print("Setting up the model and processor...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base", use_fast=True)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

print("Loading the Fakeddit dataset...")
ds = load_dataset("rtfarchitect/fakeddit_sample", split='train')
print(f"Dataset loaded with {len(ds)} samples.")

Setting up the model and processor...
cuda


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading the Fakeddit dataset...


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

multimodal_train_100k.tsv:   0%|          | 0.00/35.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Dataset loaded with 100000 samples.


In [8]:
BATCH_SIZE = 128
all_captions = []
num_batches = math.ceil(len(ds) / BATCH_SIZE)

# Use tqdm to show progress over batches, not individual images
for i in tqdm(range(num_batches), desc="Processing in Batches"):
    # A. Get a batch of URLs
    start_index = i * BATCH_SIZE
    end_index = start_index + BATCH_SIZE
    url_batch = ds['image_url'][start_index:end_index]

    # B. Prepare the image batch on the CPU
    image_batch = []
    # This list keeps track of which images in the batch succeeded or failed
    valid_indices = []
    
    for idx, url in enumerate(url_batch):
        try:
            if url and isinstance(url, str):
                image = Image.open(requests.get(url, stream=True, timeout=5).raw).convert("RGB")
                image_batch.append(image)
                valid_indices.append(idx) # Mark this index as successful
        except Exception:
            # If an image fails, we just skip it for now.
            # We'll add a None placeholder later.
            continue
            
    # If the entire batch failed to download, skip to the next batch
    if not image_batch:
        all_captions.extend([None] * len(url_batch))
        continue

    # C. Process the entire batch on the GPU in one go
    # The processor takes a list of images and creates a batch tensor
    inputs = processor(images=image_batch, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # The model generates captions for the whole batch
    outputs = blip_model.generate(**inputs, max_new_tokens=25)
    
    # Decode all captions from the batch at once
    generated_captions = processor.batch_decode(outputs, skip_special_tokens=True)

    # D. Add results back, inserting None for failed images
    batch_results = [None] * len(url_batch)
    for i, caption in zip(valid_indices, generated_captions):
        batch_results[i] = caption
    
    all_captions.extend(batch_results)

Processing in Batches:   0%|          | 0/782 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print("\nCaption generation complete. Saving to CSV...")

# Create a pandas DataFrame with the original titles and the new captions
final_df = pd.DataFrame({
    'title': ds['title'],
    'caption': all_captions
})

# Save the DataFrame to a CSV file. index=False prevents pandas from writing row numbers.
output_filename = r'/kaggle/working/fakeddit_captions.csv'
final_df.to_csv(output_filename, index=False)

print(f"✅ Successfully saved {len(final_df)} rows to {output_filename}")